# Assignment 1 — From N-Grams to Logistic Regression


## Setup

In [2]:
import math
import random
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# A. loading dataset and Pre-processing, Language Models, and Perplexity

### A1. Task contract

- **Input:** one IMDB movie review (raw text, may contain HTML)
- **Output:** 1 for positive sentiment and 0 for negative sentiment
- **Data:** balanced subset of IMDB (Hugging Face): 5,000 train (2,500 / 2,500) and 1,000 test (500 / 500)
- **Generative LM classifier:** train one bigram LM per class; label a review with the class whose LM gives the **lower perplexity**
- **Metrics:** accuracy and F1-score (positive class)
- **Rules:** only standard Python and NumPy for cleaning, features, and modeling. The Hugging Face `datasets` library is used only to download the raw data.

### A2. Load the raw dataset

We use the Hugging Face `datasets` library **only** to download the raw IMDB data . It comes with an official `train` and `test` split (25k reviews each, labels `0` = negative, `1` = positive).

In [3]:
from datasets import load_dataset

try:
    imdb = load_dataset("imdb")
except Exception:
    # Fallback to the fully qualified namespace/name identifier if 'imdb' fails to resolve
    imdb = load_dataset("stanfordnlp/imdb")

print(imdb)

raw_train_examples = [(ex["text"], int(ex["label"])) for ex in imdb["train"]]
raw_test_examples = [(ex["text"], int(ex["label"])) for ex in imdb["test"]]
raw_examples = raw_train_examples + raw_test_examples

assert all(label in {0, 1} for _, label in raw_examples), "unexpected label value"
print("Raw train reviews:", len(raw_train_examples), dict(Counter(l for _, l in raw_train_examples)))
print("Raw test reviews: ", len(raw_test_examples), dict(Counter(l for _, l in raw_test_examples)))

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
Raw train reviews: 25000 {0: 12500, 1: 12500}
Raw test reviews:  25000 {0: 12500, 1: 12500}


### A3. Inspect raw examples before choosing cleaning steps

In [4]:
for text, label in random.Random(0).sample(raw_examples, 5):
    print(f"[label={label}] {text[:350]}...\n")



[label=0] I'm trying to find something of value here. The best I can muster is that Truffaut wanted to make a movie as tedious, painful, puerile, annoying, illogical, and brainless as the experience of being in love. If that was his goal, then he succeeded, but the solution to his exercise is really a drag to watch.<br /><br />There is one scene that screams...

[label=1] Screening as part of a series of funny shorts at the Sydney Gay and Lesbian Mardi Gras film festival, this film was definitely a highlight. The script is great and the direction and acting was terrific. As another posting said, the actors' comedic timing really made this film. Lots of fun....

[label=0] This movie was, unfortunately, terrible.<br /><br />Clichéd, hackneyed, stilted dialogue and acting make it almost unwatchable.<br /><br />The feel-good finale is laughably lame.<br /><br />There is a reason Judge Reinhold's career has vanished.<br /><br />If you don't live in New York, and aren't Jewish, several of th

### A4. Build the balanced train & test subset

Exactly 2,500 + 2,500 train reviews  and 500 + 500 test reviews. Duplicates are removed first, and any test review that also appears in the train pool is dropped, so there is no leakage. The data is stored sorted by label, so we shuffle before sampling.

In [5]:
def dedupe(examples, exclude=frozenset()):
    seen, unique = set(exclude), []
    for text, label in examples:
        if text not in seen:
            seen.add(text)
            unique.append((text, label))
    return unique

def sample_balanced(examples, n_per_class, rng):
    pos = [ex for ex in examples if ex[1] == 1]
    neg = [ex for ex in examples if ex[1] == 0]
    rng.shuffle(pos)
    rng.shuffle(neg)
    chosen = pos[:n_per_class] + neg[:n_per_class]
    rng.shuffle(chosen)
    return chosen

def build_balanced_split(train_pool, test_pool, n_train_per_class=2500, n_test_per_class=500, seed=SEED):
    rng = random.Random(seed)
    train_unique = dedupe(train_pool)
    test_unique = dedupe(test_pool, exclude={t for t, _ in train_unique})
    print(f"Removed {len(train_pool) - len(train_unique)} duplicates from train pool, "
          f"{len(test_pool) - len(test_unique)} from test pool")
    return (sample_balanced(train_unique, n_train_per_class, rng),
            sample_balanced(test_unique, n_test_per_class, rng))

train_examples, test_examples = build_balanced_split(raw_train_examples, raw_test_examples)

def validate_examples(examples, name):
    assert examples, f"{name} is empty"
    for i, (text, label) in enumerate(examples):
        assert isinstance(text, str) and text.strip(), (name, i, "empty text")
        assert label in {0, 1}, (name, i, f"invalid label {label}")

validate_examples(train_examples, "train")
validate_examples(test_examples, "test")

train_counts = Counter(label for _, label in train_examples)
test_counts = Counter(label for _, label in test_examples)
print("Train size:", len(train_examples), dict(train_counts))
print("Test size: ", len(test_examples), dict(test_counts))

assert len(train_examples) == 5000 and train_counts == {1: 2500, 0: 2500}
assert len(test_examples) == 1000 and test_counts == {1: 500, 0: 500}
assert not ({t for t, _ in train_examples} & {t for t, _ in test_examples}), "train/test overlap"
print("Split checks passed (sizes, balance, no overlap).")

Removed 96 duplicates from train pool, 322 from test pool
Train size: 5000 {0: 2500, 1: 2500}
Test size:  1000 {1: 500, 0: 500}
Split checks passed (sizes, balance, no overlap).


In [6]:
def accuracy(labels, predictions):
    return float(np.mean(np.asarray(labels) == np.asarray(predictions)))

def confusion_counts(labels, predictions):
    labels, predictions = np.asarray(labels), np.asarray(predictions)
    tp = int(np.sum((labels == 1) & (predictions == 1)))
    fp = int(np.sum((labels == 0) & (predictions == 1)))
    fn = int(np.sum((labels == 1) & (predictions == 0)))
    tn = int(np.sum((labels == 0) & (predictions == 0)))
    return tp, fp, fn, tn

def f1_score(labels, predictions, positive=1):
    """F1 of the positive class (2PR / (P+R)), 0.0 when undefined."""
    labels, predictions = np.asarray(labels), np.asarray(predictions)
    if positive == 0:
        labels, predictions = 1 - labels, 1 - predictions
    tp, fp, fn, _ = confusion_counts(labels, predictions)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    return 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

y_train = np.array([label for _, label in train_examples])
y_test = np.array([label for _, label in test_examples])


# B. Pre-processing, tokenization, and vocabulary

In [7]:
HTML_TAG_RE = re.compile(r"<[^>]+>")
REPEAT_PUNCT_RE = re.compile(r"([.!?])[.!?]+")

def clean_text(text):
    text = HTML_TAG_RE.sub(" ", text)
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.lower()
    text = REPEAT_PUNCT_RE.sub(r"\1", text)
    return text

example = "This movie was GREAT!!!<br /><br />I didn\u2019t expect it... really?!"
print("Raw:    ", example)
print("Cleaned:", clean_text(example))

Raw:     This movie was GREAT!!!<br /><br />I didn’t expect it... really?!
Cleaned: this movie was great!  i didn't expect it. really?


### B1. Data Pre-processing & Tokenization (task 1.1)

Tokens are words (with optional internal apostrophe) and the punctuation marks `. ! ? ,`. Anything else (quotes, brackets, hyphens, digits-only symbols) is dropped.

**Design choice:** a review is split into **sentences** at `. ! ?`, and **each sentence** is padded with `<s>` and `</s>`. This lets the bigram LM learn how sentences start and end, and later gives next-word generation a natural stopping point. A document is therefore a list of padded sentences.

In [8]:
BOS, EOS, UNK = "<s>", "</s>", "<unk>"
SPECIALS = [BOS, EOS, UNK]
SENT_END = {".", "!", "?"}
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?|[!?.,]")

def tokenize(clean):
    return TOKEN_RE.findall(clean)

def split_sentences(tokens):
    sentences, current = [], []
    for tok in tokens:
        current.append(tok)
        if tok in SENT_END:
            sentences.append(current)
            current = []
    if current:
        sentences.append(current)
    return sentences

def preprocess(text):
    """raw review -> list of sentences, each padded as [<s>, w1, ..., wn, </s>]"""
    sentences = split_sentences(tokenize(clean_text(text)))
    return [[BOS] + s + [EOS] for s in sentences]

demo = preprocess("Not good... really?! I didn't like it, at all.")
for s in demo:
    print(s)

assert preprocess("GOOD!") == [[BOS, "good", "!", EOS]]
assert preprocess("a b. c") == [[BOS, "a", "b", ".", EOS], [BOS, "c", EOS]]
assert preprocess("<br />") == []

['<s>', 'not', 'good', '.', '</s>']
['<s>', 'really', '?', '</s>']
['<s>', 'i', "didn't", 'like', 'it', ',', 'at', 'all', '.', '</s>']


In [9]:
train_docs_raw = [preprocess(text) for text, _ in tqdm(train_examples, desc="train")]
test_docs_raw = [preprocess(text) for text, _ in tqdm(test_examples, desc="test")]


test: 100%|██████████| 1000/1000 [00:00<00:00, 7256.93it/s]



- One **shared** vocabulary for both class LMs, so both use the same `V` in the Add-1 denominator and their perplexities are comparable.
- Words seen fewer than `MIN_COUNT` times in the **whole training set** become `<unk>`. This gives `<unk>` real training counts and lets the model score unseen test words instead of failing on them.
- The test set never creates new vocabulary entries.

In [10]:
MIN_COUNT = 2

def build_lm_vocab(docs, min_count=MIN_COUNT):
    counts = Counter(tok for doc in docs for sent in doc for tok in sent)
    kept = sorted(t for t, c in counts.items() if c >= min_count and t not in SPECIALS)
    return SPECIALS + kept

def replace_oov(docs, vocab_set):
    return [[[t if t in vocab_set else UNK for t in sent] for sent in doc] for doc in docs]

itos = build_lm_vocab(train_docs_raw)
vocab_set = set(itos)
V = len(itos)

train_docs = replace_oov(train_docs_raw, vocab_set)
test_docs = replace_oov(test_docs_raw, vocab_set)


print("Vocabulary size V:", V)
print("First 10 entries:", itos[:10])

Vocabulary size V: 24126
First 10 entries: ['<s>', '</s>', '<unk>', '!', ',', '.', '0', '00', '000', '00s']


# ngram extraction

In [11]:
def extract_ngrams(tokens, n=2):
    return list(zip(*(tokens[i:] for i in range(n))))

print(extract_ngrams([BOS, "not", "good", ".", EOS], 2))

assert extract_ngrams([BOS, "a", EOS], 2) == [(BOS, "a"), ("a", EOS)]
assert extract_ngrams(["a"], 2) == []

[('<s>', 'not'), ('not', 'good'), ('good', '.'), ('.', '</s>')]


 Divide the training data by class

In [12]:

pos_docs = [doc for doc, y in zip(train_docs, y_train) if y == 1]
neg_docs = [doc for doc, y in zip(train_docs, y_train) if y == 0]

print("Positive training docs:", len(pos_docs))
print("Negative training docs:", len(neg_docs))
assert len(pos_docs) == 2500 and len(neg_docs) == 2500

Positive training docs: 2500
Negative training docs: 2500


# B2. Class-Conditional Language Models (task 1.2)

Bigrams are taken **within** each padded sentence, so no bigram ever crosses a sentence boundary (`</s>` is never a context and `<s>` is never a prediction target).

 Implement the bigram LM with Add-1 smoothing

$$P(w_i \mid w_{i-1}) = \frac{c(w_{i-1}, w_i) + 1}{c(w_{i-1}) + V}$$

where $c(w_{i-1})$ is the number of bigrams that **start** with $w_{i-1}$ (so the distribution over the next word sums to exactly 1) and $V$ is the shared vocabulary size.

Counts are stored sparsely in dictionaries because a dense $V \times V$ matrix would be far too large. Unseen bigrams get the smoothed value $1 / (c(w_{i-1}) + V)$.

In [13]:
class BigramLM:
    def __init__(self, vocab_size):
        self.V = vocab_size
        self.bigram_counts = Counter()
        self.context_counts = Counter()
        self._log_denom = {}
        self._log_unseen_ctx = math.log(self.V)   # context never seen: (0 + V)

    def fit(self, docs):
        for doc in docs:
            for sent in doc:
                self.bigram_counts.update(extract_ngrams(sent, 2))
        for (w1, _), c in self.bigram_counts.items():
            self.context_counts[w1] += c
        self._log_denom = {w1: math.log(c + self.V) for w1, c in self.context_counts.items()}
        return self

    def log_prob(self, w1, w2):
        num = math.log(self.bigram_counts.get((w1, w2), 0) + 1)
        den = self._log_denom.get(w1, self._log_unseen_ctx)
        return num - den

    def prob(self, w1, w2):
        return math.exp(self.log_prob(w1, w2))

pos_lm = BigramLM(V).fit(pos_docs)
neg_lm = BigramLM(V).fit(neg_docs)

for name, lm in [("Positive LM", pos_lm), ("Negative LM", neg_lm)]:
    print(f"{name}: {len(lm.bigram_counts):,} distinct bigrams, "
          f"{sum(lm.bigram_counts.values()):,} bigram tokens, {len(lm.context_counts):,} contexts")

Positive LM: 230,218 distinct bigrams, 687,555 bigram tokens, 21,130 contexts
Negative LM: 218,686 distinct bigrams, 671,106 bigram tokens, 20,202 contexts


### Task 3 part1
Computes the log-likelihood of a document under a given bigram LM.

    Log-likelihood = sum over all bigrams of log P(w2 | w1).
    We add logs instead of multiplying probabilities, because multiplying
    hundreds of small numbers would underflow to 0.0 on a computer.
    The result is always negative. A value closer to 0 means the model finds the document more likely.

    Computes the perplexity of a document under a given bigram LM.

    Perplexity = exp( - log-likelihood / number of bigrams ).
    Intuition: it is the average "surprise" of the model per word.
    Dividing by the number of bigrams makes long and short reviews
    comparable. LOWER perplexity means the model fits the text BETTER.

In [14]:
def log_likelihood(doc, lm):
    total = 0.0
    count = 0
    for sentence in doc:
        for w1, w2 in extract_ngrams(sentence, 2):
            total += lm.log_prob(w1, w2)
            count += 1
    return total, count


def perplexity(doc, lm):
    total, count = log_likelihood(doc, lm)
    if count == 0:
        return float("inf")
    return math.exp(-total / count)


# try it on the first 5 test reviews
for doc, label in list(zip(test_docs, y_test))[:5]:
    ll_pos, _ = log_likelihood(doc, pos_lm)
    ll_neg, _ = log_likelihood(doc, neg_lm)
    print("label:", label,
          "| LL pos:", round(ll_pos, 1), "LL neg:", round(ll_neg, 1),
          "| PP pos:", round(perplexity(doc, pos_lm), 1),
          "PP neg:", round(perplexity(doc, neg_lm), 1))

label: 1 | LL pos: -1675.1 LL neg: -1692.4 | PP pos: 1139.3 PP neg: 1225.2
label: 0 | LL pos: -1254.4 LL neg: -1249.7 | PP pos: 1748.3 PP neg: 1700.0
label: 1 | LL pos: -354.1 LL neg: -365.8 | PP pos: 1598.3 PP neg: 2041.6
label: 1 | LL pos: -1060.2 LL neg: -1096.4 | PP pos: 1659.1 PP neg: 2136.4
label: 1 | LL pos: -5746.4 LL neg: -5692.0 | PP pos: 1598.1 PP neg: 1490.4


the function examines every bigram in the document and adds its log-probability.

Here, \(D\) is the review and \(N\) is the number of tokens considered across its padded sentences.

In [15]:
def predict_lm(doc):
    pp_pos = perplexity(doc, pos_lm)
    pp_neg = perplexity(doc, neg_lm)
    if pp_pos < pp_neg:
        return 1   # positive
    return 0       # negative


y_pred_lm = []
for doc in tqdm(test_docs):
    y_pred_lm.append(predict_lm(doc))
y_pred_lm = np.array(y_pred_lm)

lm_acc = accuracy(y_test, y_pred_lm)
lm_f1 = f1_score(y_test, y_pred_lm)

print("Bigram LM accuracy:", round(lm_acc, 4))
print("Bigram LM F1 score:", round(lm_f1, 4))

100%|██████████| 1000/1000 [00:00<00:00, 2738.13it/s]

Bigram LM accuracy: 0.856
Bigram LM F1 score: 0.8545


In [16]:
# Step 1: for every word, remember which words followed it and how often
def build_followers(lm):
    followers = {}
    for (w1, w2), c in lm.bigram_counts.items():
        if w1 not in followers:
            followers[w1] = {}
        followers[w1][w2] = c
    return followers

followers_pos = build_followers(pos_lm)
followers_neg = build_followers(neg_lm)


# Step 2: keep predicting the most likely next word until </s> or max_len
def generate(prompt, followers, max_len=20):
    words = tokenize(clean_text(prompt))
    words = [w if w in vocab_set else UNK for w in words]
    context = words[-1] if len(words) > 0 else BOS

    for _ in range(max_len):
        if context not in followers:
            break

        best_word = None
        best_count = 0
        for w, c in followers[context].items():
            if w == UNK:
                continue
            if w in words and w not in [".", ",", "!", "?", EOS]:
                continue
            if c > best_count:
                best_word = w
                best_count = c

        if best_word is None or best_word == EOS:
            break

        words.append(best_word)
        context = best_word

    return " ".join(words)


print("POS:", generate("this movie was", followers_pos))
print("NEG:", generate("this movie was", followers_neg))
print("POS:", generate("i think the acting", followers_pos))
print("NEG:", generate("i think the acting", followers_neg))

POS: this movie was a great .
NEG: this movie was a good .
POS: i think the acting is a great .
NEG: i think the acting is a movie .


Goal: Given the beginning of a sentence, the model predicts the next word again and again, until it predicts the end token </s> or reaches a maximum length.

A bigram model only uses the last word as context. To continue “this movie was”, it looks only at “was” and asks which word most often came right after “was” in the training reviews. It adds that word to the sentence. The new word becomes the context, and the process repeats.



### Part 2.1: Build the vocabulary
Each review is turned into a bag of unigrams and bigrams (sentence padding tokens removed, bigrams within a sentence only). Features are counted on the training set only, and the 2,000 most frequent ones are mapped to integer indices.

In [18]:

def extract_features(text):
    # Extract unigrams and bigrams from one review.
    sentences = preprocess(text)
    features = []

    for sentence in sentences:
        # Remove sentence boundary tokens from the feature list.
        words = [w for w in sentence if w not in {BOS, EOS}]

        # Unigrams: individual tokens.
        features.extend(("UNI", word) for word in words)

        # Bigrams: consecutive tokens within each sentence.
        features.extend(
            ("BI", words[i], words[i + 1])
            for i in range(len(words) - 1)
        )

    return features


# Count unigram and bigram occurrences across training reviews only.
feature_counts = Counter()

for text, label in tqdm(train_examples, desc="Counting training features"):
    feature_counts.update(extract_features(text))

# Select the 2,000 most frequent features.
top_features = feature_counts.most_common(2000)

# Create a vocabulary: feature -> unique integer index.
vocabulary = {
    feature: index
    for index, (feature, count) in enumerate(top_features)
}

print("Total distinct features:", len(feature_counts))
print("Vocabulary size:", len(vocabulary))
print("\nFirst 10 vocabulary entries:")

for feature, index in list(vocabulary.items())[:10]:
    print(index, feature, "| count:", feature_counts[feature])

assert len(vocabulary) == 2000



Counting training features: 100%|██████████| 5000/5000 [00:01<00:00, 2731.50it/s]


Total distinct features: 437934
Vocabulary size: 2000

First 10 vocabulary entries:
0 ('UNI', 'the') | count: 67140
1 ('UNI', '.') | count: 60239
2 ('UNI', ',') | count: 55268
3 ('UNI', 'and') | count: 32926
4 ('UNI', 'a') | count: 32858
5 ('UNI', 'of') | count: 28771
6 ('UNI', 'to') | count: 27316
7 ('UNI', 'is') | count: 21541
8 ('UNI', 'in') | count: 18783
9 ('UNI', 'it') | count: 16052


### Part 2.2: Count features per class and compute Laplace-smoothed probabilities
For each class, we count how many times every vocabulary feature appears in the training reviews of that class. Features outside the 2,000-feature vocabulary are ignored. Then we convert these counts into probabilities using Add-1 (Laplace) smoothing: we add 1 to every feature count and divide by the class's total feature count plus the vocabulary size. This way, a feature that never appears in one class still gets a small non-zero probability instead of zero.

In [27]:
# Initialize feature counts for each sentiment class.
positive_counts = Counter()
negative_counts = Counter()

# Count vocabulary features separately for each class.
for text, label in tqdm(train_examples, desc="Counting class features"):
    features = extract_features(text)

    if label == 1:
        positive_counts.update(
            feature for feature in features if feature in vocabulary
        )
    else:
        negative_counts.update(
            feature for feature in features if feature in vocabulary
        )

print("Positive feature occurrences:", sum(positive_counts.values()))
print("Negative feature occurrences:", sum(negative_counts.values()))

print("\nExample counts:")
for feature in list(vocabulary.keys())[:10]:
    print(
        feature,
        "| Positive:", positive_counts[feature],
        "| Negative:", negative_counts[feature]
    )

    # Vocabulary size.
NB_V = len(vocabulary)

# Total feature occurrences in each class.
N_pos = sum(positive_counts.values())
N_neg = sum(negative_counts.values())

# Calculate Laplace-smoothed probabilities for every feature.
positive_probs = {}
negative_probs = {}

for feature in vocabulary:
    positive_probs[feature] = (
        positive_counts[feature] + 1
    ) / (N_pos + NB_V)

    negative_probs[feature] = (
        negative_counts[feature] + 1
    ) / (N_neg + NB_V)

Counting class features: 100%|██████████| 5000/5000 [00:01<00:00, 3298.96it/s]

Positive feature occurrences: 658855
Negative feature occurrences: 655197

Example counts:
('UNI', 'the') | Positive: 34651 | Negative: 32489
('UNI', '.') | Positive: 30135 | Negative: 30104
('UNI', ',') | Positive: 28786 | Negative: 26482
('UNI', 'and') | Positive: 17916 | Negative: 15010
('UNI', 'a') | Positive: 16807 | Negative: 16051
('UNI', 'of') | Positive: 15165 | Negative: 13606
('UNI', 'to') | Positive: 13358 | Negative: 13958
('UNI', 'is') | Positive: 11537 | Negative: 10004
('UNI', 'in') | Positive: 10037 | Negative: 8746
('UNI', 'it') | Positive: 8018 | Negative: 8034


### Part 2.3: Compute the log-ratios
For each feature, we take the logarithm of its positive-class probability divided by its negative-class probability. A positive value means the feature is associated with positive reviews, a negative value means it is associated with negative reviews, and a value near zero means the feature appears about equally in both classes.

In [28]:



# Calculate the log-ratio for every feature.
log_ratios = {}

for feature in vocabulary:
    log_ratios[feature] = math.log(
        positive_probs[feature] / negative_probs[feature]
    )

print("Number of positive probabilities:", len(positive_probs))
print("Number of negative probabilities:", len(negative_probs))
print("Number of log-ratios:", len(log_ratios))

assert len(positive_probs) == 2000
assert len(negative_probs) == 2000
assert len(log_ratios) == 2000

print("\nExample probabilities and log-ratios:")
for feature in list(vocabulary.keys())[:10]:
    print(
        feature,
        "| P(feature | Positive):", round(positive_probs[feature], 6),
        "| P(feature | Negative):", round(negative_probs[feature], 6),
        "| Log-ratio:", round(log_ratios[feature], 4)
    )


Number of positive probabilities: 2000
Number of negative probabilities: 2000
Number of log-ratios: 2000

Example probabilities and log-ratios:
('UNI', 'the') | P(feature | Positive): 0.052435 | P(feature | Negative): 0.049437 | Log-ratio: 0.0589
('UNI', '.') | P(feature | Positive): 0.045602 | P(feature | Negative): 0.045808 | Log-ratio: -0.0045
('UNI', ',') | P(feature | Positive): 0.04356 | P(feature | Negative): 0.040297 | Log-ratio: 0.0779
('UNI', 'and') | P(feature | Positive): 0.027112 | P(feature | Negative): 0.022841 | Log-ratio: 0.1714
('UNI', 'a') | P(feature | Positive): 0.025434 | P(feature | Negative): 0.024425 | Log-ratio: 0.0405
('UNI', 'of') | P(feature | Positive): 0.022949 | P(feature | Negative): 0.020705 | Log-ratio: 0.1029
('UNI', 'to') | P(feature | Positive): 0.020215 | P(feature | Negative): 0.02124 | Log-ratio: -0.0495
('UNI', 'is') | P(feature | Positive): 0.017459 | P(feature | Negative): 0.015224 | Log-ratio: 0.137
('UNI', 'in') | P(feature | Positive): 0.0

### Part 2.4: Naive Bayes prediction
To classify a review, we start with the log of the ratio between the positive and negative class priors. Then, for every vocabulary feature that appears in the review, we multiply its count by its log-ratio and add the result to the score. If the final score is above zero, the review is predicted as positive, otherwise it is predicted as negative. Here the prior term is zero because the training set is balanced.

In [24]:

# Calculate the class prior log-ratio.
positive_prior = np.sum(y_train == 1) / len(y_train)
negative_prior = np.sum(y_train == 0) / len(y_train)

prior_log_ratio = math.log(positive_prior / negative_prior)

print("Positive prior:", positive_prior)
print("Negative prior:", negative_prior)
print("Prior log-ratio:", prior_log_ratio)


def predict_naive_bayes(text):
    # Extract unigrams and bigrams from the review.
    features = extract_features(text)

    # Count occurrences of vocabulary features in this review.
    document_counts = Counter(
        feature for feature in features if feature in vocabulary
    )

    # Sum each feature's count multiplied by its log-ratio.
    score = prior_log_ratio

    for feature, count in document_counts.items():
        score += count * log_ratios[feature]

    # Positive if score > 0; otherwise negative.
    return 1 if score > 0 else 0


# Predict the sentiment of every test review.
y_pred_nb = []

for text, label in tqdm(test_examples, desc="Naive Bayes prediction"):
    y_pred_nb.append(predict_naive_bayes(text))

y_pred_nb = np.array(y_pred_nb)

print("\nNumber of predictions:", len(y_pred_nb))
assert len(y_pred_nb) == len(y_test) == 1000


Positive prior: 0.5
Negative prior: 0.5
Prior log-ratio: 0.0


Naive Bayes prediction: 100%|██████████| 1000/1000 [00:00<00:00, 3067.13it/s]


Number of predictions: 1000


In [25]:

nb_acc = accuracy(y_test, y_pred_nb)
nb_f1 = f1_score(y_test, y_pred_nb)

print("Naive Bayes Accuracy:", round(nb_acc, 4))
print("Naive Bayes F1-Score:", round(nb_f1, 4))



Naive Bayes Accuracy: 0.797
Naive Bayes F1-Score: 0.7931
